# C. Numerical Stability

- **C1**: Softmax exp(x - max(x)) shift
- **C2**: Cross-entropy log(p + epsilon)
- **C3**: NaN/Inf monitoring during training
- **C4**: Gradient clipping check

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from utils import softmax, cross_entropy
from network import Network, NetworkConfig
from optimizer import Adam

## C1: Softmax with Extreme Values

In [ ]:
x_large = np.array([[1000, 2000, 3000],
                     [-1000, -2000, -3000],
                     [1e10, 1e10, 1e10]])

result = softmax(x_large)
print('Softmax stability test:')
for i, row in enumerate(x_large):
    out = result[i]
    ok = not np.any(np.isnan(out)) and not np.any(np.isinf(out))
    print(f'  Input:  {row}')
    print(f'  Output: {out}')
    print(f'  Sum:    {out.sum():.10f} | NaN: {np.any(np.isnan(out))} | {"PASS" if ok else "FAIL"}')
    print()

## C2: Cross-Entropy Boundary Values

In [ ]:
y_true = np.array([[1, 0], [0, 1], [1, 0]])

cases = [
    ('Normal',     np.array([[0.9, 0.1], [0.2, 0.8], [0.7, 0.3]])),
    ('Near zero',  np.array([[0.999999, 1e-15], [1e-15, 0.999999], [0.999999, 1e-15]])),
    ('Exact 0/1',  np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 0.0]])),
]

for name, y_pred in cases:
    loss = cross_entropy(y_true, y_pred)
    ok = not np.isnan(loss) and not np.isinf(loss)
    print(f'{name:12s} -> loss: {loss:.6f} | {"PASS" if ok else "FAIL"}')

## C3: NaN/Inf Monitoring During Training

In [ ]:
np.random.seed(42)

config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
optimizer = Adam(learning_rate=0.001)

x = np.random.randn(100, 30)
y = np.zeros((100, 2))
y[np.arange(100), np.random.randint(0, 2, 100)] = 1

nan_found = False
for epoch in range(200):
    output = net.forward(x)
    loss = net.loss(y, output)
    nw, nb = net.backward(y)
    optimizer.update(net, nw, nb)

    # Check loss
    if np.isnan(loss) or np.isinf(loss):
        print(f'Epoch {epoch}: NaN/Inf in LOSS'); nan_found = True; break

    # Check gradients
    for i, (gw, gb) in enumerate(zip(nw, nb)):
        if np.any(np.isnan(gw)) or np.any(np.isnan(gb)):
            print(f'Epoch {epoch}: NaN in gradient layer {i}'); nan_found = True
        if np.any(np.isinf(gw)) or np.any(np.isinf(gb)):
            print(f'Epoch {epoch}: Inf in gradient layer {i}'); nan_found = True

    # Check weights
    for i, (w, b) in enumerate(zip(net.weights, net.biases)):
        if np.any(np.isnan(w)) or np.any(np.isnan(b)):
            print(f'Epoch {epoch}: NaN in weights layer {i}'); nan_found = True

    if nan_found: break

if not nan_found:
    print(f'No NaN/Inf detected over 200 epochs. Final loss: {loss:.6f}')
    print('Status: PASS')
else:
    print('Status: FAIL')

## C4: Gradient Magnitude Check

In [ ]:
np.random.seed(42)

config = NetworkConfig(layers=[30, 24, 24, 24, 2], activation='relu',
    loss='cross_entropy', output_activation='softmax', weights_initializer='heUniform')
net = Network(config)
optimizer = Adam(learning_rate=0.001)

x = np.random.randn(100, 30)
y = np.zeros((100, 2))
y[np.arange(100), np.random.randint(0, 2, 100)] = 1

grad_norms = {i: [] for i in range(len(net.weights))}

for epoch in range(200):
    net.forward(x)
    nw, nb = net.backward(y)
    optimizer.update(net, nw, nb)
    for i, gw in enumerate(nw):
        grad_norms[i].append(np.linalg.norm(gw))

plt.figure(figsize=(10, 5))
for i, norms in grad_norms.items():
    plt.plot(norms, label=f'Layer {i}')
plt.title('Gradient Norms per Layer')
plt.xlabel('Epoch'); plt.ylabel('Gradient Norm')
plt.legend(); plt.grid(True); plt.show()

threshold = 100.0
for i, norms in grad_norms.items():
    mx = max(norms)
    print(f'Layer {i}: max grad norm = {mx:.4f} | {"OK" if mx < threshold else "EXPLODING"}')

needs_clip = any(max(n) > threshold for n in grad_norms.values())
print(f'\nGradient clipping needed: {"YES" if needs_clip else "NO"}')